# Отчёт по рыночным факторам — этап 1: загрузка данных

Ноутбук скачивает все показатели из спецификации, складывает их в дневные таблицы по категориям и сохраняет в Excel.

**Как устроен ноутбук** (идём сверху вниз, ячейка за ячейкой):

1. Библиотеки и настройки периода
2. **Спецификация** — единственное место, которое ты редактируешь: *что* загрузить
3. Знакомство с API: как выглядит «сырой» ответ ЦБ и Мосбиржи
4. Функции загрузки: *как* скачать данные из каждого источника (пишутся один раз)
5. Загрузка по спецификации, расчётные показатели, проверка, сохранение

Месячные агрегаты, сводка и сравнение м/м и с начала года — этап 2, следующий ноутбук.

Установка библиотек (один раз, в терминале или в ячейке с `!` в начале):
```
pip install requests pandas matplotlib openpyxl lxml
```

## 1. Библиотеки

- `requests` — отправляет HTTP-запросы (по сути, «открывает ссылку» из кода);
- `pandas` — таблицы и временные ряды;
- `xml.etree.ElementTree` — разбор XML (в этом формате отвечает ЦБ);
- `StringIO` — позволяет передать pandas текст так, будто это файл.

In [ ]:
import xml.etree.ElementTree as ET
from io import StringIO

import matplotlib.pyplot as plt
import pandas as pd
import requests

# Некоторые сайты (в т.ч. ЦБ) не любят запросы без «подписи браузера»
HEADERS = {"User-Agent": "Mozilla/5.0"}

## 2. Настройки периода

Указываешь только **отчётный месяц**. Окно загрузки считается автоматически: 13 месяцев назад от отчётного. Этого хватает и на графики за год, и на сравнение «с начала года» (нужно значение на конец декабря), и на м/м.

In [ ]:
REPORT_MONTH = "2026-09"   # отчётный месяц, ГГГГ-ММ
WINDOW_MONTHS = 13         # сколько месяцев истории загружать

report_period = pd.Period(REPORT_MONTH, freq="M")
DATE_FROM = (report_period - WINDOW_MONTHS).start_time.strftime("%Y-%m-%d")
DATE_TILL = report_period.end_time.strftime("%Y-%m-%d")

print(f"Загружаем данные с {DATE_FROM} по {DATE_TILL}")

## 3. Спецификация — что загружаем

Это декларативная часть: каждая строка описывает **что** нужно, а не **как** это скачать. Чтобы добавить показатель, достаточно дописать строку.

**`SERIES`** — временные ряды. Колонки:
- **название** — как показатель будет называться в таблицах;
- **категория** — на какой лист Excel (а потом и слайд) он попадёт;
- **источник** — каким загрузчиком качать (`cbr_soap`, `cbr_currency`, `moex_index`) или `curve`, если ряд вырезается из кривой;
- **код** — идентификатор в источнике (метод ЦБ, тикер индекса, код валюты, название кривой);
- **поле** — какой столбец взять из ответа (у одного индекса есть и значение `CLOSE`, и доходность `YIELD`);
- **единица**.

**`CURVES`** — кривые КБД и ROISfix. Хранятся отдельно, потому что это «срок → ставка на каждую дату», а не один ряд.

**`DERIVED`** — расчётные показатели. Формула пишется по названиям из `SERIES` в обратных кавычках, почти как в Excel. Умножение на 100 переводит разницу в процентах в базисные пункты.

In [ ]:
#           название              категория          источник        код              поле        ед.
SERIES = [
    ("USD/RUB",             "Макро",           "cbr_currency", "R01235",        "Value",    "руб."),

    ("Ключевая ставка",     "Денежный рынок",  "cbr_soap",     "KeyRateXML",    "Rate",     "%"),
    ("RUONIA",              "Денежный рынок",  "cbr_soap",     "RuoniaXML",     "ruo",      "%"),
    ("RUSFAR ON",           "Денежный рынок",  "moex_index",   "RUSFAR",        "CLOSE",    "%"),
    ("Ликвидность",         "Денежный рынок",  "cbr_soap",     "BliquidityXML", "StrLiDef", "млрд руб."),
    ("ROISfix 3M",          "Денежный рынок",  "curve",        "ROISfix",       "R3M",      "%"),
    ("ROISfix 1Y",          "Денежный рынок",  "curve",        "ROISfix",       "R1Y",      "%"),

    ("КБД 1 год",           "Облигации",       "curve",        "КБД",           "1",        "%"),
    ("КБД 10 лет",          "Облигации",       "curve",        "КБД",           "10",       "%"),
    ("RGBI",                "Облигации",       "moex_index",   "RGBI",          "CLOSE",    "пункты"),
    ("Доходность RGBI",     "Облигации",       "moex_index",   "RGBI",          "YIELD",    "%"),
    ("Корпораты AAA",       "Облигации",       "moex_index",   "RUCBCPAAANS",   "CLOSE",    "пункты"),
    ("Доходность AAA",      "Облигации",       "moex_index",   "RUCBCPAAANS",   "YIELD",    "%"),
    ("Корпораты A",         "Облигации",       "moex_index",   "RUCBCPANS",     "CLOSE",    "пункты"),
    ("Доходность A",        "Облигации",       "moex_index",   "RUCBCPANS",     "YIELD",    "%"),
]

#           название     источник     код
CURVES = [
    ("КБД",       "cbr_zcyc",  "zcyc"),
    ("ROISfix",   "cbr_soap",  "ROISfixXML"),
]

#           название                    категория          формула                                          ед.
DERIVED = [
    ("Спред RUONIA – ключевая",   "Денежный рынок",  "(`RUONIA` - `Ключевая ставка`) * 100",          "б.п."),
    ("Спред RUSFAR – ключевая",   "Денежный рынок",  "(`RUSFAR ON` - `Ключевая ставка`) * 100",       "б.п."),
    ("Наклон КБД 10л – 1г",       "Облигации",       "(`КБД 10 лет` - `КБД 1 год`) * 100",            "б.п."),
    ("Спред AAA – ОФЗ",           "Облигации",       "(`Доходность AAA` - `Доходность RGBI`) * 100",  "б.п."),
    ("Спред A – AAA",             "Облигации",       "(`Доходность A` - `Доходность AAA`) * 100",     "б.п."),
]

> **Что пока упрощено и что проверим первым запуском.**
> - Спред AAA – ОФЗ здесь считается как простая разница доходностей индексов. На этапе 2 заменим его на корректный: доходность корпоративного индекса минус КБД на сроке его дюрации.
> - Названия полей ЦБ (`ruo`, `StrLiDef`, `R3M`, `R1Y`) и наличие `YIELD` у корпоративных индексов я не мог проверить отсюда: сайты ЦБ и Мосбиржи закрыты для моего инструмента. Если где-то поле называется иначе, загрузчик не упадёт молча, а напечатает список доступных полей. Поправить нужно будет одно слово в строке спецификации.

## 4. Знакомство с API

Прежде чем прятать загрузку в функции, посмотрим, что на самом деле присылают серверы. Это две ячейки-«экскурсии»: в рабочем процессе их можно не запускать.

### 4.1. ЦБ: веб-сервис DailyInfo (протокол SOAP)

SOAP — старый, но надёжный формат. Мы **отправляем** на адрес сервиса XML-«конверт» с названием метода и параметрами, а сервис **возвращает** XML-конверт с данными.

- адрес у всех методов один: `https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx` (если открыть его в браузере, увидишь список всех методов);
- какой метод вызвать, говорят заголовок `SOAPAction` и тег внутри конверта;
- параметры периода у всех нужных нам методов одинаковые: `fromDate` и `ToDate`.

Отправляем запрос ключевой ставки за две недели и печатаем начало ответа:

In [ ]:
envelope = '''<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <KeyRateXML xmlns="http://web.cbr.ru/">
      <fromDate>2026-09-01T00:00:00</fromDate>
      <ToDate>2026-09-14T00:00:00</ToDate>
    </KeyRateXML>
  </soap:Body>
</soap:Envelope>'''

response = requests.post(
    "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx",
    data=envelope.encode("utf-8"),
    headers={**HEADERS,
             "Content-Type": "text/xml; charset=utf-8",
             "SOAPAction": '"http://web.cbr.ru/KeyRateXML"'},
    timeout=60,
)

print("Статус ответа:", response.status_code)   # 200 = всё хорошо
print(response.text[:1500])

В ответе видна матрёшка: `Envelope → Body → KeyRateXMLResponse → KeyRateXMLResult → ...`, а внутри повторяющиеся записи, в каждой дата и значение. Так устроены все XML-методы DailyInfo, меняются только названия тегов. Поэтому один универсальный загрузчик справится со всеми.

### 4.2. Мосбиржа: ISS (обычный REST-API в формате JSON)

Здесь проще: данные запрашиваются обычной ссылкой с параметрами, как в браузере. Пример ссылки, которую можно открыть руками:
`https://iss.moex.com/iss/history/engines/stock/markets/index/securities/RGBI.json?from=2026-09-01`

Путь читается как «история → торговая система stock → рынок index → инструмент RGBI». Ответ — JSON, то есть вложенные словари. Нужный блок называется `history`: в нём список названий столбцов `columns` и строки `data`.

In [ ]:
response = requests.get(
    "https://iss.moex.com/iss/history/engines/stock/markets/index/securities/RGBI.json",
    params={"from": "2026-09-01", "till": "2026-09-14", "iss.meta": "off"},
    timeout=60,
)
answer = response.json()

print("Блоки в ответе:", list(answer.keys()))
print("Столбцы:", answer["history"]["columns"])
pd.DataFrame(answer["history"]["data"], columns=answer["history"]["columns"]).head()

Посмотри на столбцы: помимо `CLOSE` там должны быть `YIELD` (доходность) и `DURATION` (дюрация). Поэтому в спецификации у RGBI две строки с одним кодом, но разными полями.

Важная особенность ISS: **за один запрос он отдаёт не больше 100 строк**. Для длинного периода нужно запрашивать «страницы» параметром `start` (0, 100, 200…), пока ответ не станет пустым. Это сделано в загрузчике ниже.

## 5. Функции загрузки

Здесь живёт «как». Каждая функция принимает одинаковые аргументы `(код, дата_с, дата_по)` и возвращает таблицу pandas, где **индекс — даты, а столбцы — поля ответа**. Благодаря одинаковому формату основной цикл ниже не знает и не должен знать, откуда пришли данные.

### 5.1. ЦБ, SOAP — универсальный загрузчик

Шаги:
1. собрать конверт с названием метода и датами;
2. отправить его POST-запросом;
3. найти в ответе тег `<Метод>Result`;
4. внутри него лежит контейнер с повторяющимися записями: каждая запись становится строкой таблицы, её дочерние теги — столбцами;
5. первый тег записи — всегда дата, она становится индексом.

In [ ]:
def cbr_soap(method, date_from, date_till):
    envelope = f'''<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <{method} xmlns="http://web.cbr.ru/">
      <fromDate>{date_from}T00:00:00</fromDate>
      <ToDate>{date_till}T00:00:00</ToDate>
    </{method}>
  </soap:Body>
</soap:Envelope>'''

    response = requests.post(
        "https://www.cbr.ru/DailyInfoWebServ/DailyInfo.asmx",
        data=envelope.encode("utf-8"),
        headers={**HEADERS,
                 "Content-Type": "text/xml; charset=utf-8",
                 "SOAPAction": f'"http://web.cbr.ru/{method}"'},
        timeout=60,
    )
    response.raise_for_status()   # если сервер ответил ошибкой — остановиться с понятным сообщением

    root = ET.fromstring(response.content)
    result = next(el for el in root.iter() if el.tag.endswith(method + "Result"))
    records = result[0]           # контейнер с повторяющимися записями

    rows = []
    for record in records:
        # тег вида "{namespace}Rate" превращаем в "Rate"
        rows.append({child.tag.split("}")[-1]: child.text for child in record})
    table = pd.DataFrame(rows)

    date_column = table.columns[0]
    table[date_column] = pd.to_datetime(table[date_column].str[:10])
    table = table.set_index(date_column).sort_index()
    table.index.name = "date"
    return table.apply(pd.to_numeric, errors="coerce")

### 5.2. Мосбиржа, история индекса — с постраничной загрузкой

Цикл `while` запрашивает страницу за страницей и сдвигает `start` на число полученных строк. Пустая страница означает, что данные закончились.

In [ ]:
def moex_index(secid, date_from, date_till):
    url = f"https://iss.moex.com/iss/history/engines/stock/markets/index/securities/{secid}.json"
    rows, columns, start = [], None, 0

    while True:
        params = {"from": date_from, "till": date_till, "start": start, "iss.meta": "off"}
        block = requests.get(url, params=params, timeout=60).json()["history"]
        if not block["data"]:
            break
        rows.extend(block["data"])
        columns = block["columns"]
        start += len(block["data"])

    if not rows:
        print(f"   Мосбиржа не вернула данных по {secid}")
        return pd.DataFrame()

    table = pd.DataFrame(rows, columns=columns)
    table["TRADEDATE"] = pd.to_datetime(table["TRADEDATE"])
    table = table.drop_duplicates("TRADEDATE", keep="last").set_index("TRADEDATE").sort_index()
    table.index.name = "date"
    return table

### 5.3. ЦБ, курс валюты (XML-выгрузка)

Это не SOAP, а простая ссылка, которая возвращает XML. Курс записан с запятой (`83,2500`), поэтому меняем её на точку.

In [ ]:
def cbr_currency(currency_code, date_from, date_till):
    params = {
        "date_req1": pd.Timestamp(date_from).strftime("%d/%m/%Y"),
        "date_req2": pd.Timestamp(date_till).strftime("%d/%m/%Y"),
        "VAL_NM_RQ": currency_code,
    }
    response = requests.get("https://www.cbr.ru/scripts/XML_dynamic.asp",
                            params=params, headers=HEADERS, timeout=60)
    root = ET.fromstring(response.content)

    rows = []
    for record in root.iter("Record"):
        rows.append({
            "date": pd.to_datetime(record.get("Date"), format="%d.%m.%Y"),
            "Value": float(record.find("Value").text.replace(",", ".")),
        })
    return pd.DataFrame(rows).set_index("date").sort_index()

### 5.4. ЦБ, КБД (HTML-таблица)

ЦБ публикует КБД таблицей на странице: строки — даты, столбцы — сроки от 0,25 до 30 лет. Отдельного API тут нет, зато `pandas.read_html` превращает таблицу со страницы в DataFrame одной строкой. У таблицы двухэтажная шапка («Срок до погашения, лет» → сроки), поэтому названия столбцов берём из нижнего этажа и приводим к виду `"0.25"`, `"1"`, `"10"`.

In [ ]:
def cbr_zcyc(_code, date_from, date_till):
    params = {
        "UniDbQuery.Posted": "True",
        "UniDbQuery.From": pd.Timestamp(date_from).strftime("%d.%m.%Y"),
        "UniDbQuery.To": pd.Timestamp(date_till).strftime("%d.%m.%Y"),
    }
    response = requests.get("https://www.cbr.ru/hd_base/zcyc_params/",
                            params=params, headers=HEADERS, timeout=60)
    table = pd.read_html(StringIO(response.text), decimal=",", thousands=" ")[0]

    tenors = []
    for column in table.columns[1:]:
        label = column[-1] if isinstance(column, tuple) else column
        tenors.append(f"{float(str(label).replace(',', '.')):g}")   # "0,25" -> "0.25", "10.0" -> "10"

    table.columns = ["date"] + tenors
    table["date"] = pd.to_datetime(table["date"], format="%d.%m.%Y")
    return table.set_index("date").sort_index()

### 5.5. Справочник загрузчиков

Словарь связывает название источника из спецификации с функцией. Когда в `SERIES` написано `"moex_index"`, цикл возьмёт отсюда функцию `moex_index`. Новый источник = новая функция + одна строка здесь.

In [ ]:
LOADERS = {
    "cbr_soap":     cbr_soap,
    "cbr_currency": cbr_currency,
    "moex_index":   moex_index,
    "cbr_zcyc":     cbr_zcyc,
}

## 6. Загрузка по спецификации

### 6.1. Сначала кривые

Кривые грузим раньше рядов, потому что часть рядов (КБД 1 год, ROISfix 3M) из них вырезается.

In [ ]:
curves = {}
for name, source, code_ in CURVES:
    print(f"Кривая: {name}")
    curves[name] = LOADERS[source](code_, DATE_FROM, DATE_TILL)
    print(f"   получено {len(curves[name])} дат, сроки: {list(curves[name].columns)}")

### 6.2. Ряды

Логика цикла:
- если источник `curve`, берём столбец из уже загруженной кривой;
- иначе скачиваем ответ источника, но **один раз на код**: RGBI нужен дважды (значение и доходность), и во второй раз он возьмётся из словаря `raw`, без нового запроса;
- если нужного поля в ответе нет, печатаем, какие поля есть, и идём дальше.

In [ ]:
raw = {}      # кэш ответов: (источник, код) -> таблица
series = {}   # итог: название -> ряд

for name, category, source, code_, field, unit in SERIES:
    if source == "curve":
        table = curves[code_]
    else:
        if (source, code_) not in raw:
            print(f"Загружаю {source}: {code_}")
            raw[(source, code_)] = LOADERS[source](code_, DATE_FROM, DATE_TILL)
        table = raw[(source, code_)]

    if field not in table.columns:
        print(f"   ⚠ {name}: поля '{field}' нет. Доступные поля: {list(table.columns)}")
        continue

    series[name] = pd.to_numeric(table[field], errors="coerce")

print(f"\nЗагружено рядов: {len(series)} из {len(SERIES)}")

### 6.3. Общая дневная таблица и расчётные показатели

Склеиваем все ряды по датам. У источников разные календари: ЦБ устанавливает курс на субботу, биржа торгует не во все дни, ключевая ставка публикуется не каждый день. Поэтому после склейки протягиваем последнее известное значение вперёд (`ffill`): значение действует, пока не появится следующее. Без этого спреды считались бы только в те дни, когда оба источника случайно совпали.

Затем по формулам из `DERIVED` считаем спреды методом `eval`.

In [ ]:
daily = pd.DataFrame(series).sort_index()
daily = daily.loc[DATE_FROM:DATE_TILL].ffill()

for name, category, formula, unit in DERIVED:
    try:
        daily[name] = daily.eval(formula)
    except Exception as error:
        print(f"⚠ Не удалось посчитать «{name}»: {error}")

daily.tail()

## 7. Проверка

Таблица-паспорт: по каждому показателю видно, с какой и по какую дату пришли данные, последнее значение и число пустых дней. Пустой ряд или ряд, обрывающийся раньше конца периода, сразу бросится в глаза.

In [ ]:
units = {row[0]: row[5] for row in SERIES} | {row[0]: row[3] for row in DERIVED}

check = pd.DataFrame({
    "ед.":            pd.Series(units),
    "первая дата":    daily.apply(lambda s: s.first_valid_index()),
    "последняя дата": daily.apply(lambda s: s.last_valid_index()),
    "последнее":      daily.ffill().iloc[-1].round(2),
    "пустых дней":    daily.isna().sum(),
})
check

Быстрые графики по категориям — проверить глазами, что ряды выглядят правдоподобно (нет провалов в ноль, нет скачков из-за не того поля). Это ещё не графики отчёта.

In [ ]:
categories = sorted({row[1] for row in SERIES} | {row[1] for row in DERIVED})

for category in categories:
    columns = [row[0] for row in SERIES + DERIVED if row[1] == category and row[0] in daily]
    daily[columns].plot(subplots=True, figsize=(11, 2.2 * len(columns)),
                        title=[f"{c}, {units[c]}" for c in columns], legend=False, grid=True)
    plt.suptitle(category, y=1.0, fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Сохранение в Excel

Один лист на категорию плюс по листу на каждую кривую. Этот файл — вход для этапа 2 (месячные агрегаты, сводка, м/м и с начала года).

In [ ]:
OUTPUT_FILE = f"data_daily_{REPORT_MONTH}.xlsx"

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    for category in categories:
        columns = [row[0] for row in SERIES + DERIVED if row[1] == category and row[0] in daily]
        daily[columns].to_excel(writer, sheet_name=category)
    for name, table in curves.items():
        table.loc[DATE_FROM:DATE_TILL].to_excel(writer, sheet_name=f"Кривая {name}")

print(f"Сохранено: {OUTPUT_FILE}")